# Human Action and Biomechanics Video Analysis Pipeline

This notebook provides a complete, production-ready pipeline for analyzing human actions, biomechanics, and grip force estimation from video.

## Features
- **MediaPipe Holistic**: 33-point pose and 21-point hand landmark detection
- **YOLOv8**: Real-time object detection and tracking
- **VideoMAE**: Action recognition using transformer-based models
- **Physics-Based Grip Force Estimation**: Calculate grip forces based on object properties
- **Biomechanics Analysis**: Joint angles, velocities, and movement patterns

## Usage
1. Run all cells sequentially
2. Upload your video or use the sample video
3. Review the analysis results and visualizations

## Cell 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q ultralytics mediapipe opencv-python pandas numpy matplotlib
!pip install -q transformers[torch] accelerate

print("✓ All dependencies installed successfully")

## Cell 2: Import Libraries and Initialize Models

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import json
import torch
from tqdm import tqdm

# MediaPipe
import mediapipe as mp
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# YOLOv8
from ultralytics import YOLO

# VideoMAE for action recognition
from transformers import AutoImageProcessor, AutoModelForVideoClassification

print("✓ All libraries imported successfully")

# Initialize MediaPipe Holistic
holistic = mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=1
)

# Initialize YOLOv8
print("Loading YOLOv8 model...")
yolo_model = YOLO('yolov8n.pt')  # nano model for speed
print("✓ YOLOv8 model loaded")

# Initialize VideoMAE
print("Loading VideoMAE model...")
image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
videomae_model = AutoModelForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
videomae_model.eval()
print("✓ VideoMAE model loaded")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
videomae_model.to(device)
print(f"✓ Using device: {device}")

## Cell 3: Physics-Based Grip Force Estimator

In [ ]:
class PhysicsGripEstimator:
    """
    Physics-based grip force estimator using material properties.
    
    Calculates minimum and maximum grip forces based on:
    - Object mass and friction coefficient (minimum to prevent slipping)
    - Puncture and distributed failure forces (maximum before damage)
    """
    
    def __init__(self):
        """Initialize with default material properties database."""
        self.materials_db = {
            "egg": {
                "mass_kg": 0.05,
                "friction_coef": 0.3,
                "puncture_force_n": 25.0,
                "distributed_fail_n": 60.0
            },
            "wine glass": {
                "mass_kg": 0.3,
                "friction_coef": 0.4,
                "puncture_force_n": 15.0,
                "distributed_fail_n": 200.0
            },
            "banana": {
                "mass_kg": 0.12,
                "friction_coef": 0.5,
                "puncture_force_n": 8.0,
                "distributed_fail_n": 40.0
            },
            "cell phone": {
                "mass_kg": 0.2,
                "friction_coef": 0.6,
                "puncture_force_n": 500.0,
                "distributed_fail_n": 1000.0
            },
            "screwdriver": {
                "mass_kg": 0.15,
                "friction_coef": 0.5,
                "puncture_force_n": 300.0,
                "distributed_fail_n": 500.0
            },
            "hammer": {
                "mass_kg": 0.8,
                "friction_coef": 0.5,
                "puncture_force_n": 200.0,
                "distributed_fail_n": 2000.0
            },
            "tennis racket": {
                "mass_kg": 0.3,
                "friction_coef": 0.6,
                "puncture_force_n": 50.0,
                "distributed_fail_n": 500.0
            },
            "baseball": {
                "mass_kg": 0.145,
                "friction_coef": 0.5,
                "puncture_force_n": 100.0,
                "distributed_fail_n": 800.0
            },
            "cup": {
                "mass_kg": 0.25,
                "friction_coef": 0.4,
                "puncture_force_n": 30.0,
                "distributed_fail_n": 300.0
            },
            "spatula": {
                "mass_kg": 0.1,
                "friction_coef": 0.5,
                "puncture_force_n": 100.0,
                "distributed_fail_n": 400.0
            },
            "bowl": {
                "mass_kg": 0.4,
                "friction_coef": 0.4,
                "puncture_force_n": 40.0,
                "distributed_fail_n": 400.0
            },
            "__default__": {
                "mass_kg": 0.5,
                "friction_coef": 0.5,
                "puncture_force_n": 100.0,
                "distributed_fail_n": 500.0
            }
        }
    
    def get_force_bounds(self, object_name: str, finger_count: float = 2.5) -> tuple:
        """
        Calculate grip force bounds for a given object.
        
        Args:
            object_name: Name of the object (case-insensitive)
            finger_count: Number of fingers involved in grip (default 2.5 for thumb+index+partial middle)
        
        Returns:
            tuple: (min_force_N, max_force_N, warning_N)
                - min_force: Minimum force to prevent slipping
                - max_force: Maximum force before damage
                - warning: 80% of max_force as warning threshold
        """
        # Normalize object name
        object_name_lower = object_name.lower().strip()
        
        # Get material properties
        if object_name_lower in self.materials_db:
            props = self.materials_db[object_name_lower]
        else:
            props = self.materials_db["__default__"]
        
        mass = props["mass_kg"]
        friction = props["friction_coef"]
        puncture = props["puncture_force_n"]
        distributed = props["distributed_fail_n"]
        
        # Calculate minimum force (to prevent slipping)
        # F_min = (m * g) / (μ * n_fingers)
        g = 9.81  # gravitational acceleration m/s²
        min_force = (mass * g) / (friction * finger_count)
        
        # Calculate maximum force (before damage)
        # F_max = min(puncture_force * n_fingers, distributed_fail_force)
        max_force = min(puncture_force * finger_count, distributed)
        
        # Warning threshold at 80% of max
        warning = max_force * 0.8
        
        return (min_force, max_force, warning)
    
    def add_object(self, name: str, mass_kg: float, friction_coef: float, 
                   puncture_force_n: float, distributed_fail_n: float) -> None:
        """
        Add a new object to the materials database.
        
        Args:
            name: Object name
            mass_kg: Mass in kilograms
            friction_coef: Coefficient of static friction
            puncture_force_n: Local force to puncture/break surface (Newtons)
            distributed_fail_n: Force to crush entire object (Newtons)
        """
        self.materials_db[name.lower().strip()] = {
            "mass_kg": mass_kg,
            "friction_coef": friction_coef,
            "puncture_force_n": puncture_force_n,
            "distributed_fail_n": distributed_fail_n
        }
        print(f"✓ Added '{name}' to materials database")
    
    def save_database(self, filepath: str = "materials_database.json") -> None:
        """
        Export the materials database to a JSON file.
        
        Args:
            filepath: Path to save the JSON file
        """
        with open(filepath, 'w') as f:
            json.dump(self.materials_db, f, indent=2)
        print(f"✓ Database saved to {filepath}")
    
    def load_database(self, filepath: str) -> None:
        """
        Load materials database from a JSON file.
        
        Args:
            filepath: Path to the JSON file
        """
        with open(filepath, 'r') as f:
            self.materials_db = json.load(f)
        print(f"✓ Database loaded from {filepath}")
    
    def list_objects(self) -> list:
        """Return list of all objects in the database."""
        return [k for k in self.materials_db.keys() if k != "__default__"]


# Initialize the physics engine
physics_engine = PhysicsGripEstimator()
print(f"✓ PhysicsGripEstimator initialized with {len(physics_engine.list_objects())} objects")
print(f"Available objects: {', '.join(physics_engine.list_objects())}")

## Cell 4: Analysis Helper Functions

In [ ]:
def calculate_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    Calculate the angle at point b formed by points a-b-c.
    
    Args:
        a: First point (x, y)
        b: Vertex point (x, y)
        c: Third point (x, y)
    
    Returns:
        float: Angle in degrees (0-180)
    """
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    c = np.array(c, dtype=np.float32)
    
    # Vectors
    ba = a - b
    bc = c - b
    
    # Calculate angle using dot product
    dot_product = np.dot(ba, bc)
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    
    if norm_ba == 0 or norm_bc == 0:
        return 0.0
    
    cos_angle = np.clip(dot_product / (norm_ba * norm_bc), -1.0, 1.0)
    angle_rad = np.arccos(cos_angle)
    angle_deg = np.degrees(angle_rad)
    
    return angle_deg


def calculate_velocity(prev_pos: np.ndarray, curr_pos: np.ndarray, fps: float) -> float:
    """
    Calculate velocity between two positions.
    
    Args:
        prev_pos: Previous position (x, y)
        curr_pos: Current position (x, y)
        fps: Frames per second of the video
    
    Returns:
        float: Velocity in pixels/second
    """
    if prev_pos is None or curr_pos is None:
        return 0.0
    
    prev_pos = np.array(prev_pos, dtype=np.float32)
    curr_pos = np.array(curr_pos, dtype=np.float32)
    
    displacement = np.linalg.norm(curr_pos - prev_pos)
    time_delta = 1.0 / fps if fps > 0 else 1.0
    
    velocity = displacement / time_delta
    
    return velocity


def estimate_force_from_grip(hand_landmarks, object_name: str, img_w: int, img_h: int, 
                             physics_engine: PhysicsGripEstimator,
                             max_grip_px: float = 200, min_grip_px: float = 15) -> tuple:
    """
    Estimate grip force from hand landmarks and object properties.
    
    Args:
        hand_landmarks: MediaPipe hand landmarks
        object_name: Name of the detected object
        img_w: Image width in pixels
        img_h: Image height in pixels
        physics_engine: PhysicsGripEstimator instance
        max_grip_px: Maximum grip distance (open hand) in pixels
        min_grip_px: Minimum grip distance (closed fist) in pixels
    
    Returns:
        tuple: (force_newtons, status_string, pixel_distance)
            - force_newtons: Estimated grip force
            - status_string: "SAFE", "WARNING", "DANGER", "SLIP_RISK", or "NO_HAND"
            - pixel_distance: Distance between thumb and index fingertips
    """
    if hand_landmarks is None or not hand_landmarks.landmark:
        return (0.0, "NO_HAND", 0.0)
    
    # Get thumb tip (landmark 4) and index fingertip (landmark 8)
    thumb_tip = hand_landmarks.landmark[4]
    index_tip = hand_landmarks.landmark[8]
    
    # Convert normalized coordinates to pixel coordinates
    thumb_px = np.array([thumb_tip.x * img_w, thumb_tip.y * img_h])
    index_px = np.array([index_tip.x * img_w, index_tip.y * img_h])
    
    # Calculate pixel distance
    pixel_distance = np.linalg.norm(thumb_px - index_px)
    
    # Get physics bounds for the object
    min_force, max_force, warning_force = physics_engine.get_force_bounds(object_name)
    
    # Map pixel distance inversely to force
    # Smaller distance = higher force (tighter grip)
    # Clamp pixel distance to valid range
    pixel_distance_clamped = np.clip(pixel_distance, min_grip_px, max_grip_px)
    
    # Normalize to 0-1 range (0 = fully open, 1 = fully closed)
    grip_normalized = (max_grip_px - pixel_distance_clamped) / (max_grip_px - min_grip_px)
    
    # Map to force range
    # At grip_normalized=0, force = min_force (just holding)
    # At grip_normalized=1, force = max_force (maximum safe grip)
    force_newtons = min_force + grip_normalized * (max_force - min_force)
    
    # Determine status
    if force_newtons >= warning_force:
        status = "DANGER"
    elif force_newtons >= min_force * 1.5:
        status = "WARNING"
    elif force_newtons < min_force:
        status = "SLIP_RISK"
    else:
        status = "SAFE"
    
    return (force_newtons, status, pixel_distance)


print("✓ Helper functions defined:")
print("  - calculate_angle(a, b, c)")
print("  - calculate_velocity(prev_pos, curr_pos, fps)")
print("  - estimate_force_from_grip(hand_landmarks, object_name, img_w, img_h, physics_engine, max_grip_px, min_grip_px)")

## Cell 5: Configuration Variables

In [ ]:
# ==================== CONFIGURATION ====================

# Frame processing
FRAME_SKIP = 1              # Process every Nth frame (1 = all frames)
YOLO_EVERY_N = 5            # Run YOLO detection every N frames

# Action recognition
ACTION_CLIP_LEN = 16        # Number of frames for VideoMAE input
ACTION_INTERVAL = 30        # Run action recognition every N frames

# Grip force estimation
MAX_GRIP_PX = 200           # Maximum grip distance (open hand) in pixels
MIN_GRIP_PX = 15            # Minimum grip distance (closed fist) in pixels

# Visualization
SHOW_VISUALIZATION = True   # Display real-time visualization
SAVE_OUTPUT_VIDEO = True    # Save annotated output video

# ======================================================

print("Configuration loaded:")
print(f"  FRAME_SKIP: {FRAME_SKIP}")
print(f"  YOLO_EVERY_N: {YOLO_EVERY_N}")
print(f"  ACTION_CLIP_LEN: {ACTION_CLIP_LEN}")
print(f"  ACTION_INTERVAL: {ACTION_INTERVAL}")
print(f"  MAX_GRIP_PX: {MAX_GRIP_PX}")
print(f"  MIN_GRIP_PX: {MIN_GRIP_PX}")

## Cell 6: Main Video Processing Class

In [ ]:
class VideoAnalysisPipeline:
    """
    Complete video analysis pipeline integrating:
    - MediaPipe Holistic for pose and hand landmarks
    - YOLOv8 for object detection
    - VideoMAE for action recognition
    - Physics-based grip force estimation
    """
    
    def __init__(self, physics_engine: PhysicsGripEstimator):
        """
        Initialize the video analysis pipeline.
        
        Args:
            physics_engine: PhysicsGripEstimator instance
        """
        self.physics_engine = physics_engine
        self.frame_buffer = deque(maxlen=ACTION_CLIP_LEN)
        self.results_data = []
        self.prev_left_wrist_pos = None
        self.current_objects = []
        self.primary_object = "unknown"
        self.last_action_label = "unknown"
        self.last_action_confidence = 0.0
        
    def run_yolo_detection(self, frame: np.ndarray) -> tuple:
        """
        Run YOLOv8 object detection on a frame.
        
        Args:
            frame: Input frame (BGR)
        
        Returns:
            tuple: (primary_object_name, all_detected_objects)
        """
        results = yolo_model(frame, verbose=False)
        
        detected_objects = []
        primary_object = "unknown"
        max_confidence = 0.0
        
        if results[0].boxes is not None:
            boxes = results[0].boxes
            for i in range(len(boxes)):
                conf = float(boxes.conf[i])
                class_id = int(boxes.cls[i])
                class_name = yolo_model.names[class_id]
                
                detected_objects.append({
                    "name": class_name,
                    "confidence": conf,
                    "bbox": boxes.xyxy[i].cpu().numpy().tolist()
                })
                
                if conf > max_confidence:
                    max_confidence = conf
                    primary_object = class_name
        
        if not detected_objects:
            primary_object = "unknown"
        
        return (primary_object, detected_objects)
    
    def run_videomae_inference(self) -> tuple:
        """
        Run VideoMAE action recognition on the frame buffer.
        
        Returns:
            tuple: (action_label, confidence)
        """
        if len(self.frame_buffer) < ACTION_CLIP_LEN:
            return ("buffering", 0.0)
        
        # Convert buffer to list of frames
        frames_list = list(self.frame_buffer)
        
        # Prepare input for VideoMAE
        inputs = image_processor(frames_list, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Run inference
        with torch.no_grad():
            outputs = videomae_model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)[0]
            predicted_class_idx = torch.argmax(probabilities).item()
            confidence = float(probabilities[predicted_class_idx])
            action_label = videomae_model.config.id2label[predicted_class_idx]
        
        return (action_label, confidence)
    
    def process_frame(self, frame: np.ndarray, frame_idx: int, fps: float) -> dict:
        """
        Process a single frame through the complete pipeline.
        
        Args:
            frame: Input frame (BGR)
            frame_idx: Frame index
            fps: Frames per second
        
        Returns:
            dict: Frame analysis results
        """
        img_h, img_w = frame.shape[:2]
        timestamp = frame_idx / fps if fps > 0 else 0.0
        
        # Convert BGR to RGB for MediaPipe
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Run MediaPipe Holistic
        mp_results = holistic.process(frame_rgb)
        
        # Extract pose landmarks
        pose_landmarks = mp_results.pose_landmarks
        left_hand_landmarks = mp_results.left_hand_landmarks
        right_hand_landmarks = mp_results.right_hand_landmarks
        
        # Calculate left elbow angle (shoulder-elbow-wrist)
        left_elbow_angle = 0.0
        if pose_landmarks:
            shoulder = [pose_landmarks.landmark[11].x, pose_landmarks.landmark[11].y]
            elbow = [pose_landmarks.landmark[13].x, pose_landmarks.landmark[13].y]
            wrist = [pose_landmarks.landmark[15].x, pose_landmarks.landmark[15].y]
            left_elbow_angle = calculate_angle(shoulder, elbow, wrist)
        
        # Calculate left wrist velocity
        left_wrist_velocity = 0.0
        current_left_wrist_pos = None
        if pose_landmarks:
            current_left_wrist_pos = np.array([
                pose_landmarks.landmark[15].x * img_w,
                pose_landmarks.landmark[15].y * img_h
            ])
            
            if self.prev_left_wrist_pos is not None:
                left_wrist_velocity = calculate_velocity(
                    self.prev_left_wrist_pos, 
                    current_left_wrist_pos, 
                    fps
                )
            
            self.prev_left_wrist_pos = current_left_wrist_pos.copy()
        
        # Grip force estimation (using left hand by default)
        grip_force_n = 0.0
        grip_status = "NO_HAND"
        grip_pixel_dist = 0.0
        left_hand_present = False
        
        if left_hand_landmarks:
            left_hand_present = True
            grip_force_n, grip_status, grip_pixel_dist = estimate_force_from_grip(
                left_hand_landmarks,
                self.primary_object,
                img_w,
                img_h,
                self.physics_engine,
                MAX_GRIP_PX,
                MIN_GRIP_PX
            )
        
        # Add frame to buffer for VideoMAE (resize to 224x224)
        frame_resized = cv2.resize(frame, (224, 224))
        frame_rgb_resized = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        self.frame_buffer.append(frame_rgb_resized)
        
        # Return frame data
        frame_data = {
            "frame": frame_idx,
            "timestamp": round(timestamp, 3),
            "detected_primary_object": self.primary_object,
            "all_objects": str(self.current_objects),  # Store as string for DataFrame
            "left_elbow_angle": round(left_elbow_angle, 2),
            "left_wrist_velocity": round(left_wrist_velocity, 2),
            "grip_force_n": round(grip_force_n, 2),
            "grip_status": grip_status,
            "grip_pixel_dist": round(grip_pixel_dist, 2),
            "left_hand_present": left_hand_present,
            "action_label": self.last_action_label,
            "action_confidence": round(self.last_action_confidence, 3)
        }
        
        return frame_data, mp_results
    
    def analyze_video(self, video_path: str, output_path: str = "output_analysis.mp4") -> pd.DataFrame:
        """
        Process entire video through the analysis pipeline.
        
        Args:
            video_path: Path to input video
            output_path: Path to save annotated output video
        
        Returns:
            pd.DataFrame: Complete analysis results
        """
        # Open video
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")
        
        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        print(f"Video properties:")
        print(f"  Resolution: {img_w}x{img_h}")
        print(f"  FPS: {fps}")
        print(f"  Total frames: {total_frames}")
        
        # Setup output video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = None
        if SAVE_OUTPUT_VIDEO:
            out = cv2.VideoWriter(output_path, fourcc, fps, (img_w, img_h))
        
        # Reset state
        self.results_data = []
        self.frame_buffer.clear()
        self.prev_left_wrist_pos = None
        
        # Progress bar
        pbar = tqdm(total=total_frames, desc="Processing video")
        
        frame_idx = 0
        
        while True:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # Skip frames if configured
            if frame_idx % FRAME_SKIP != 0:
                frame_idx += 1
                pbar.update(1)
                continue
            
            # Run YOLO detection periodically
            if frame_idx % YOLO_EVERY_N == 0:
                self.primary_object, self.current_objects = self.run_yolo_detection(frame)
            
            # Process frame through pipeline
            frame_data, mp_results = self.process_frame(frame, frame_idx, fps)
            
            # Run VideoMAE inference periodically
            if frame_idx % ACTION_INTERVAL == 0 and frame_idx > 0:
                action_label, action_conf = self.run_videomae_inference()
                self.last_action_label = action_label
                self.last_action_confidence = action_conf
                # Update current frame data with new action
                frame_data["action_label"] = action_label
                frame_data["action_confidence"] = round(action_conf, 3)
            
            # Store results
            self.results_data.append(frame_data)
            
            # Visualization
            if SHOW_VISUALIZATION or SAVE_OUTPUT_VIDEO:
                vis_frame = self.visualize_frame(frame, mp_results, frame_data)
                
                if SAVE_OUTPUT_VIDEO and out is not None:
                    out.write(vis_frame)
                
                if SHOW_VISUALIZATION and frame_idx % 10 == 0:
                    # Show every 10th frame to avoid overwhelming display
                    plt.figure(figsize=(12, 8))
                    plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
                    plt.axis('off')
                    plt.tight_layout()
                    plt.show()
            
            frame_idx += 1
            pbar.update(1)
        
        # Cleanup
        pbar.close()
        cap.release()
        if out is not None:
            out.release()
        
        # Create DataFrame
        df = pd.DataFrame(self.results_data)
        
        print(f"\n✓ Analysis complete!")
        print(f"  Processed {len(df)} frames")
        if SAVE_OUTPUT_VIDEO:
            print(f"  Output video saved to: {output_path}")
        
        return df
    
    def visualize_frame(self, frame: np.ndarray, mp_results, frame_data: dict) -> np.ndarray:
        """
        Create annotated visualization of analysis results.
        
        Args:
            frame: Input frame (BGR)
            mp_results: MediaPipe results
            frame_data: Frame analysis data
        
        Returns:
            np.ndarray: Annotated frame
        """
        vis_frame = frame.copy()
        
        # Draw pose landmarks
        if mp_results.pose_landmarks:
            mp_drawing.draw_landmarks(
                vis_frame,
                mp_results.pose_landmarks,
                mp_holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)
            )
        
        # Draw hand landmarks
        if mp_results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                vis_frame,
                mp_results.left_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 255), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 128, 0), thickness=2)
            )
        
        if mp_results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                vis_frame,
                mp_results.right_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 255), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 128, 0), thickness=2)
            )
        
        # Add text overlays
        y_offset = 30
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        
        # Primary object
        cv2.putText(vis_frame, f"Object: {frame_data['detected_primary_object']}", 
                   (10, y_offset), font, font_scale, (0, 255, 0), 2)
        
        # Elbow angle
        cv2.putText(vis_frame, f"Elbow Angle: {frame_data['left_elbow_angle']:.1f}°",
                   (10, y_offset + 30), font, font_scale, (255, 255, 0), 2)
        
        # Wrist velocity
        cv2.putText(vis_frame, f"Wrist Velocity: {frame_data['left_wrist_velocity']:.1f} px/s",
                   (10, y_offset + 60), font, font_scale, (255, 255, 0), 2)
        
        # Grip force
        grip_color = (0, 255, 0)  # Green for SAFE
        if frame_data['grip_status'] == "WARNING":
            grip_color = (0, 255, 255)  # Yellow
        elif frame_data['grip_status'] in ["DANGER", "SLIP_RISK"]:
            grip_color = (0, 0, 255)  # Red
        
        cv2.putText(vis_frame, f"Grip Force: {frame_data['grip_force_n']:.1f}N [{frame_data['grip_status']}]",
                   (10, y_offset + 90), font, font_scale, grip_color, 2)
        
        # Action label
        cv2.putText(vis_frame, f"Action: {frame_data['action_label']} ({frame_data['action_confidence']:.2f})",
                   (10, y_offset + 120), font, font_scale, (255, 0, 255), 2)
        
        # Frame info
        cv2.putText(vis_frame, f"Frame: {frame_data['frame']}",
                   (10, y_offset + 150), font, 0.5, (255, 255, 255), 1)
        
        return vis_frame


print("✓ VideoAnalysisPipeline class defined")

## Cell 7: Sample Video Download (Optional)

In [ ]:
#@title Download Sample Video (Optional)
#@markdown If you don't have a video to analyze, run this cell to download a sample video.

import os
import urllib.request

SAMPLE_VIDEO_URL = "https://media.blender.org/wp-content/uploads/2019/05/Spring_thumbnail_1280.jpg"

# For demonstration, we'll create a simple test video
def create_test_video(output_path="test_video.mp4", duration_sec=5, fps=30):
    """Create a simple test video with moving shapes."""
    import random
    
    img_w, img_h = 640, 480
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (img_w, img_h))
    
    total_frames = duration_sec * fps
    
    print(f"Creating test video: {total_frames} frames...")
    
    for frame_idx in tqdm(range(total_frames)):
        frame = np.zeros((img_h, img_w, 3), dtype=np.uint8)
        
        # Draw a moving circle (simulating a hand)
        x = int(img_w * 0.5 + 100 * np.sin(frame_idx * 0.1))
        y = int(img_h * 0.5 + 50 * np.cos(frame_idx * 0.15))
        cv2.circle(frame, (x, y), 30, (0, 255, 0), -1)
        
        # Draw a static rectangle (simulating an object)
        cv2.rectangle(frame, (400, 200), (550, 350), (0, 0, 255), -1)
        
        # Add frame number
        cv2.putText(frame, f"Frame: {frame_idx}", (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        out.write(frame)
    
    out.release()
    print(f"✓ Test video created: {output_path}")
    return output_path

# Create test video
test_video_path = create_test_video()
print(f"\nTest video ready at: {test_video_path}")

## Cell 8: Upload Your Video

In [ ]:
#@title Upload Your Video
#@markdown Upload a video file from your local machine for analysis.

from google.colab import files
import os

print("Please upload a video file...")
uploaded = files.upload()

# Get the uploaded video path
video_path = None
for filename in uploaded.keys():
    if filename.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        video_path = filename
        print(f"✓ Uploaded: {filename} ({os.path.getsize(filename)} bytes)")
        break

if video_path is None:
    print("No valid video file uploaded. Using test video instead.")
    video_path = "test_video.mp4"
else:
    print(f"\nVideo ready for analysis: {video_path}")

## Cell 9: Run Video Analysis

In [ ]:
#@title Run Complete Video Analysis
#@markdown Execute the full analysis pipeline on your video.

# Initialize the pipeline
pipeline = VideoAnalysisPipeline(physics_engine)

# Run analysis
print(f"Starting analysis of: {video_path}")
print("="*60)

results_df = pipeline.analyze_video(
    video_path=video_path,
    output_path="annotated_output.mp4"
)

print("\n" + "="*60)
print("Analysis Results Summary:")
print(f"  Total frames analyzed: {len(results_df)}")
print(f"  Unique objects detected: {results_df['detected_primary_object'].nunique()}")
print(f"  Unique actions recognized: {results_df['action_label'].nunique()}")
print(f"  Average grip force: {results_df['grip_force_n'].mean():.2f}N")
print(f"  Average elbow angle: {results_df['left_elbow_angle'].mean():.2f}°")
print("="*60)

## Cell 10: View Analysis Results

In [ ]:
#@title Display Analysis Results
#@markdown View the complete analysis results as a DataFrame.

# Display first few rows
print("First 10 frames:")
display(results_df.head(10))

# Display statistics
print("\nStatistical Summary:")
display(results_df.describe())

# Check for different grip statuses
print("\nGrip Status Distribution:")
print(results_df['grip_status'].value_counts())

# Check detected objects
print("\nDetected Objects:")
print(results_df['detected_primary_object'].value_counts().head(10))

# Check recognized actions
print("\nRecognized Actions:")
print(results_df['action_label'].value_counts().head(10))

## Cell 11: Visualize Results

In [ ]:
#@title Generate Analysis Visualizations
#@markdown Create plots showing biomechanics metrics over time.

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Plot 1: Left Elbow Angle Over Time
axes[0, 0].plot(results_df['timestamp'], results_df['left_elbow_angle'], 'b-', linewidth=1)
axes[0, 0].set_xlabel('Time (seconds)')
axes[0, 0].set_ylabel('Angle (degrees)')
axes[0, 0].set_title('Left Elbow Angle Over Time')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=90, color='r', linestyle='--', alpha=0.5, label='90°')
axes[0, 0].legend()

# Plot 2: Left Wrist Velocity Over Time
axes[0, 1].plot(results_df['timestamp'], results_df['left_wrist_velocity'], 'g-', linewidth=1)
axes[0, 1].set_xlabel('Time (seconds)')
axes[0, 1].set_ylabel('Velocity (px/s)')
axes[0, 1].set_title('Left Wrist Velocity Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Grip Force Over Time
axes[1, 0].plot(results_df['timestamp'], results_df['grip_force_n'], 'r-', linewidth=1)
axes[1, 0].set_xlabel('Time (seconds)')
axes[1, 0].set_ylabel('Force (Newtons)')
axes[1, 0].set_title('Estimated Grip Force Over Time')
axes[1, 0].grid(True, alpha=0.3)

# Color-code by grip status
status_colors = {'SAFE': 'green', 'WARNING': 'orange', 'DANGER': 'red', 'SLIP_RISK': 'purple', 'NO_HAND': 'gray'}
for status in results_df['grip_status'].unique():
    mask = results_df['grip_status'] == status
    axes[1, 0].scatter(results_df[mask]['timestamp'], results_df[mask]['grip_force_n'], 
                      c=status_colors.get(status, 'gray'), label=status, s=10, alpha=0.5)
axes[1, 0].legend(loc='upper right')

# Plot 4: Grip Pixel Distance Over Time
axes[1, 1].plot(results_df['timestamp'], results_df['grip_pixel_dist'], 'm-', linewidth=1)
axes[1, 1].set_xlabel('Time (seconds)')
axes[1, 1].set_ylabel('Distance (pixels)')
axes[1, 1].set_title('Thumb-Index Finger Distance Over Time')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=MIN_GRIP_PX, color='r', linestyle='--', alpha=0.5, label='Min Grip')
axes[1, 1].axhline(y=MAX_GRIP_PX, color='g', linestyle='--', alpha=0.5, label='Max Grip')
axes[1, 1].legend()

# Plot 5: Action Confidence Over Time
axes[2, 0].plot(results_df['timestamp'], results_df['action_confidence'], 'c-', linewidth=1)
axes[2, 0].set_xlabel('Time (seconds)')
axes[2, 0].set_ylabel('Confidence')
axes[2, 0].set_title('Action Recognition Confidence Over Time')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].set_ylim(0, 1)

# Plot 6: Elbow Angle Histogram
axes[2, 1].hist(results_df['left_elbow_angle'], bins=30, color='blue', alpha=0.7, edgecolor='black')
axes[2, 1].set_xlabel('Angle (degrees)')
axes[2, 1].set_ylabel('Frequency')
axes[2, 1].set_title('Distribution of Left Elbow Angles')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analysis_plots.png', dpi=150, bbox_inches='tight')
print("✓ Plots saved to 'analysis_plots.png'")
plt.show()

## Cell 12: Export Results

In [ ]:
#@title Export Analysis Results
#@markdown Save analysis results to CSV and download.

# Save to CSV
csv_filename = "video_analysis_results.csv"
results_df.to_csv(csv_filename, index=False)
print(f"✓ Results saved to {csv_filename}")

# Save materials database
physics_engine.save_database("materials_database.json")

# Download files
print("\nDownloading files...")
files.download(csv_filename)
files.download("materials_database.json")

if SAVE_OUTPUT_VIDEO and os.path.exists("annotated_output.mp4"):
    files.download("annotated_output.mp4")

if os.path.exists("analysis_plots.png"):
    files.download("analysis_plots.png")

print("\n✓ All files downloaded!")

## Cell 13: Advanced Usage Examples

In [ ]:
#@title Advanced: Custom Object Properties
#@markdown Add custom objects to the physics database with specific material properties.

# Example: Add a custom object
physics_engine.add_object(
    name="water bottle",
    mass_kg=0.5,           # 500ml water bottle
    friction_coef=0.6,     # Plastic surface
    puncture_force_n=150.0,  # Hard to puncture
    distributed_fail_n=800.0  # Can withstand significant crushing
)

# Example: Add another custom object
physics_engine.add_object(
    name="apple",
    mass_kg=0.18,
    friction_coef=0.45,
    puncture_force_n=20.0,
    distributed_fail_n=100.0
)

# Get force bounds for custom objects
print("Custom object force bounds:")
for obj in ["water bottle", "apple"]:
    min_f, max_f, warn_f = physics_engine.get_force_bounds(obj)
    print(f"  {obj}: Min={min_f:.2f}N, Max={max_f:.2f}N, Warning={warn_f:.2f}N")

# Updated object list
print(f"\nTotal objects in database: {len(physics_engine.list_objects())}")
print(f"Objects: {', '.join(physics_engine.list_objects())}")

In [ ]:
#@title Advanced: Analyze Specific Metrics
#@markdown Perform detailed analysis on specific biomechanics metrics.

# Find frames with high grip force
high_grip_frames = results_df[results_df['grip_force_n'] > results_df['grip_force_n'].quantile(0.9)]
print(f"Frames with top 10% grip force: {len(high_grip_frames)}")
if len(high_grip_frames) > 0:
    print(high_grip_frames[['frame', 'timestamp', 'grip_force_n', 'grip_status', 'detected_primary_object']].head())

# Find frames with extreme elbow angles
extreme_angles = results_df[(results_df['left_elbow_angle'] < 45) | (results_df['left_elbow_angle'] > 150)]
print(f"\nFrames with extreme elbow angles (<45° or >150°): {len(extreme_angles)}")

# Calculate movement smoothness (jerk)
if len(results_df) > 2:
    results_df['velocity_change'] = results_df['left_wrist_velocity'].diff()
    results_df['jerk'] = results_df['velocity_change'].diff()
    avg_jerk = results_df['jerk'].abs().mean()
    print(f"\nAverage jerk (movement smoothness): {avg_jerk:.2f} px/s²")
    print("Lower jerk = smoother movements")

# Action transitions
print("\nAction label transitions:")
action_changes = results_df[results_df['action_label'] != results_df['action_label'].shift(1)]
print(f"Number of action transitions: {len(action_changes) - 1}")
if len(action_changes) > 1:
    print("Sample transitions:")
    for idx in range(min(5, len(action_changes)-1)):
        prev_action = action_changes['action_label'].iloc[idx]
        next_action = action_changes['action_label'].iloc[idx+1] if idx+1 < len(action_changes) else "N/A"
        print(f"  Frame {action_changes['frame'].iloc[idx]}: {prev_action} → {next_action}")

## Cell 14: Cleanup and Resources

In [ ]:
#@title Cleanup Resources
#@markdown Release resources and clean up temporary files.

import gc
import torch

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared")

# Release MediaPipe resources
holistic.close()
print("✓ MediaPipe resources released")

# Force garbage collection
gc.collect()
print("✓ Garbage collection completed")

print("\n" + "="*60)
print("PIPELINE EXECUTION COMPLETE")
print("="*60)
print("\nGenerated files:")
print("  - video_analysis_results.csv: Complete frame-by-frame analysis")
print("  - materials_database.json: Object properties database")
print("  - annotated_output.mp4: Video with visualizations")
print("  - analysis_plots.png: Biomechanics metric plots")
print("\nThank you for using the Human Action Analysis Pipeline!")
print("="*60)